**Running Total** and **Rolling Total** are both window function techniques in SQL, but they solve different problems.

| Feature     | Running Total                                           | Rolling Total                                |
| ----------- | ------------------------------------------------------- | -------------------------------------------- |
| Definition  | Cumulative sum from the beginning up to the current row | Sum over a fixed-size moving window          |
| Window Size | Grows with each row                                     | Fixed number of rows or time period          |
| Use Cases   | Bank balance, cumulative sales, total profit            | 7-day sales, 3-month average, moving metrics |

---

# 1. Running Total (Cumulative Sum)

A **running total** continuously adds values from the first row until the current row.

### Example Table: Sales

| SaleDate | Amount |
| -------- | ------ |
| 1-Jan    | 100    |
| 2-Jan    | 200    |
| 3-Jan    | 150    |
| 4-Jan    | 300    |

We want:

| SaleDate | Amount | RunningTotal |
| -------- | ------ | ------------ |
| 1-Jan    | 100    | 100          |
| 2-Jan    | 200    | 300          |
| 3-Jan    | 150    | 450          |
| 4-Jan    | 300    | 750          |

### SQL

```sql
SELECT
    SaleDate,
    Amount,
    SUM(Amount) OVER (
        ORDER BY SaleDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTotal
FROM Sales;
```

### Explanation

```sql
SUM(Amount) OVER (...)
```

This tells SQL to calculate the sum as a **window function**.

```sql
ORDER BY SaleDate
```

Calculates values in date order.

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

Means:

* Start from the first row
* Continue until the current row

So SQL performs:

```
Row1 = 100

Row2 = 100 + 200 = 300

Row3 = 100 + 200 + 150 = 450

Row4 = 100 + 200 + 150 + 300 = 750
```

---

# 2. Rolling Total (Moving Total)

A **rolling total** only considers a fixed number of rows (or a time period).

Example: Last **3 days** of sales.

### Same Data

| SaleDate | Amount |
| -------- | ------ |
| 1-Jan    | 100    |
| 2-Jan    | 200    |
| 3-Jan    | 150    |
| 4-Jan    | 300    |
| 5-Jan    | 250    |

Desired output:

| Date  | Amount | Rolling 3 Days |
| ----- | ------ | -------------- |
| 1-Jan | 100    | 100            |
| 2-Jan | 200    | 300            |
| 3-Jan | 150    | 450            |
| 4-Jan | 300    | 650            |
| 5-Jan | 250    | 700            |

Notice:

For **4-Jan**

```
200 + 150 + 300 = 650
```

The first day's value (100) is no longer included.

---

### SQL

```sql
SELECT
    SaleDate,
    Amount,
    SUM(Amount) OVER (
        ORDER BY SaleDate
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS Rolling3Days
FROM Sales;
```

---

### Explanation

```sql
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
```

Means

Current row +

Previous 2 rows

For each row:

```
Row1
100

Row2
100 + 200 = 300

Row3
100 + 200 +150 =450

Row4
200 +150 +300 =650

Row5
150 +300 +250 =700
```

The window keeps moving down the table.

---

# Visual Difference

Running Total

```
100
100+200
100+200+150
100+200+150+300
100+200+150+300+250
```

Rolling Total (3 rows)

```
100
100+200
100+200+150
200+150+300
150+300+250
```

Notice the window slides forward.

---

# Using `PARTITION BY`

Suppose sales are stored by employee.

| Employee | Date  | Sales |
| -------- | ----- | ----- |
| A        | 1-Jan | 100   |
| A        | 2-Jan | 200   |
| A        | 3-Jan | 150   |
| B        | 1-Jan | 300   |
| B        | 2-Jan | 100   |

Running total should restart for each employee.

```sql
SELECT
    Employee,
    SaleDate,
    Sales,
    SUM(Sales) OVER (
        PARTITION BY Employee
        ORDER BY SaleDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTotal
FROM Sales;
```

Result

| Employee | Date  | Sales | RunningTotal |
| -------- | ----- | ----- | ------------ |
| A        | 1-Jan | 100   | 100          |
| A        | 2-Jan | 200   | 300          |
| A        | 3-Jan | 150   | 450          |
| B        | 1-Jan | 300   | 300          |
| B        | 2-Jan | 100   | 400          |

Each employee gets an independent cumulative total.

---

# Real-World Uses

### Running Total

* Bank account balance
* Total sales to date
* Inventory accumulation
* Cumulative profit
* Year-to-date (YTD) revenue

Example:

```
Jan 1000
Feb 1500
Mar 1200

Running Total:
1000
2500
3700
```

---

### Rolling Total

* Last 7 days' sales
* Last 30 days' revenue
* Moving average
* Rolling profit
* Stock market analysis

Example:

```
Daily Sales

100
120
140
200
150
```

Rolling 3-day totals:

```
100
220
360
460
490
```

---

# Key Window Frame Clauses

| Clause                | Meaning                                   |
| --------------------- | ----------------------------------------- |
| `UNBOUNDED PRECEDING` | Start from the first row in the partition |
| `CURRENT ROW`         | Include the current row                   |
| `2 PRECEDING`         | Include the two previous rows             |
| `1 FOLLOWING`         | Include the next row                      |
| `UNBOUNDED FOLLOWING` | Continue to the last row in the partition |

---

## Summary

* **Running Total**: A cumulative sum from the beginning to the current row. Implement it with:

  ```sql
  SUM(column) OVER (
      ORDER BY column
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  )
  ```
* **Rolling Total**: A moving sum over a fixed-size window (such as the current row and the previous two rows). Implement it with:

  ```sql
  SUM(column) OVER (
      ORDER BY column
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
  )
  ```
* Use **`PARTITION BY`** when you want the calculation to restart for each group (for example, per employee, customer, or product).


In [0]:
from pyspark.sql import Row

data = [
    Row(Product="Laptop", SaleDate="2026-07-01", Amount=1200, Store="NY", Salesperson="Alice"),
    Row(Product="Laptop", SaleDate="2026-07-02", Amount=1350, Store="NY", Salesperson="Bob"),
    Row(Product="Laptop", SaleDate="2026-07-03", Amount=1100, Store="LA", Salesperson="Charlie"),
    Row(Product="Phone", SaleDate="2026-07-01", Amount=800, Store="NY", Salesperson="Alice"),
    Row(Product="Phone", SaleDate="2026-07-02", Amount=950, Store="LA", Salesperson="Charlie"),
    Row(Product="Phone", SaleDate="2026-07-03", Amount=900, Store="NY", Salesperson="Bob"),
    Row(Product="Tablet", SaleDate="2026-07-01", Amount=600, Store="LA", Salesperson="Charlie"),
    Row(Product="Tablet", SaleDate="2026-07-02", Amount=650, Store="NY", Salesperson="Alice"),
    Row(Product="Tablet", SaleDate="2026-07-03", Amount=700, Store="LA", Salesperson="Bob"),
    Row(Product="Monitor", SaleDate="2026-07-01", Amount=300, Store="NY", Salesperson="Alice"),
    Row(Product="Monitor", SaleDate="2026-07-02", Amount=350, Store="LA", Salesperson="Charlie"),
    Row(Product="Monitor", SaleDate="2026-07-03", Amount=400, Store="NY", Salesperson="Bob"),
]

sales_df = spark.createDataFrame(data)
sales_df.createOrReplaceTempView("sales_df")
display(sales_df)

In [0]:
%sql
select product,
saledate,
amount,
sum(amount) over(partition by product order by saledate) as running_total_by_product,
sum(amount) over( order by saledate) as total_sale_date

from sales_df order by product,saledate

In [0]:
%sql
select product,
saledate,
amount,
SUM(amount) OVER (
    ORDER BY saledate
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) as running_total

from sales_df

## Rolling Total:
 A moving sum over a fixed-size window (such as the current row and the previous two rows). Implement it with:

In [0]:
%sql
select product,
saledate,
amount,
SUM(amount) OVER (
    ORDER BY saledate
    ROWs between 2 preceding and current row
) as rolling_sum_by3

from sales_df 